# Phase 3B — GPU Training (Google Colab T4)
**Target: ≥95% validation accuracy**

### Before you start:
1. Upload these 2 files to your Google Drive (anywhere, e.g. `MyDrive/mushroom/`):
   - `dataset_augmented.zip`
   - `phase3b_train.py`
2. Runtime → **Change runtime type** → **T4 GPU** → Save
3. Run cells top-to-bottom

Expected time: **~2 hours** on T4. Keep the tab open.

## Cell 1 — Check GPU

In [ ]:
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))
# Must show GPU here. If empty → Runtime > Change runtime type > T4 GPU

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Set your Drive folder path
> **Edit `DRIVE_FOLDER` to wherever you uploaded the files.**

In [ ]:
import os, shutil

# ── EDIT THIS to match your Drive folder ────────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/mushroom'
# ────────────────────────────────────────────────────────────────────────

WORK_DIR = '/content/mushroom_project'
os.makedirs(WORK_DIR, exist_ok=True)

# Copy phase3b_train.py to working dir
shutil.copy(f'{DRIVE_FOLDER}/phase3b_train.py', WORK_DIR)

os.chdir(WORK_DIR)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

## Cell 4 — Unzip dataset
> Unzips directly from Drive. Takes ~2–3 minutes.

In [ ]:
ZIP_PATH = f'{DRIVE_FOLDER}/dataset_augmented.zip'

if not os.path.isdir('dataset_augmented'):
    print('Unzipping dataset_augmented.zip ...')
    !unzip -q "{ZIP_PATH}" -d .
    print('Done!')
else:
    print('dataset_augmented/ already exists, skipping unzip.')

# Verify structure
for split in ['train', 'val', 'test']:
    path = f'dataset_augmented/{split}'
    if os.path.isdir(path):
        total = sum(len(os.listdir(f'{path}/{c}')) for c in os.listdir(path) if os.path.isdir(f'{path}/{c}'))
        print(f'  {split:5s}: {total} images in {os.listdir(path)}')
    else:
        print(f'  WARNING: {path} not found!')

## Cell 5 — Install packages

In [ ]:
!pip install -q scikit-learn matplotlib
print('Done.')

## Cell 6 — Run training
> **This takes ~2 hours. Do NOT close this tab or let the session idle.**
>
> Tip: click somewhere in the page every 30 min to prevent idle disconnect.

In [ ]:
# If you get OOM (out of memory) error, uncomment the next 2 lines:
# import phase3b_train
# phase3b_train.BATCH_SIZE = 16

%run phase3b_train.py

## Cell 7 — Results

In [ ]:
import json
with open('phase3b_results.json') as f:
    results = json.load(f)

print(f'\n{"Model":<25} {"Val(raw)":>9} {"Val(TTA)":>9} {"Test(TTA)":>10}')
print('-' * 57)
for r in results:
    print(f"{r['model']:<25}"
          f"  {r.get('val_accuracy', 0)*100:>7.2f}%"
          f"  {r.get('val_acc_tta',  0)*100:>7.2f}%"
          f"  {r.get('test_acc_tta', 0)*100:>8.2f}%")

In [ ]:
# Show training curves
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('plots/*_curves.png')):
    print(p); display(Image(p))

In [ ]:
# Show confusion matrices
for p in sorted(glob.glob('plots/*_cm.png')):
    print(p); display(Image(p))

## Cell 8 — Copy results back to Drive

In [ ]:
import shutil, os

SAVE_DIR = f'{DRIVE_FOLDER}/phase3b_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy models
shutil.copytree('models', f'{SAVE_DIR}/models', dirs_exist_ok=True)
# Copy plots
shutil.copytree('plots',  f'{SAVE_DIR}/plots',  dirs_exist_ok=True)
# Copy reports
for f in ['phase3b_report.txt', 'phase3b_results.json']:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)

print('Saved to Drive:', SAVE_DIR)
print('Contents:', os.listdir(SAVE_DIR))